# Importing the necessary libraries

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
import numpy as np
import optuna
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading the datasets

In [28]:
blr_df = pd.read_csv('/kaggle/input/spatial-computing-project-dataset-1/blr_df.csv')
hyd_df = pd.read_csv('/kaggle/input/spatial-computing-project-dataset-1/hyd_df.csv')
pune_df = pd.read_csv('/kaggle/input/spatial-computing-project-dataset-1/pune_df.csv')

In [29]:
blr_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,12.498648,916.589625,0.608896,327.505071,93.194473,30.939528
1,2003-01-02,12.825784,918.004624,3.006925,290.310937,93.578218,31.179113
2,2003-01-03,12.970981,918.453619,2.872020,279.921010,94.227231,33.045705
3,2003-01-04,12.445614,917.966755,2.678422,274.952584,93.297114,38.266933
4,2003-01-05,14.071074,918.579933,3.104962,275.257170,94.353249,33.185394
...,...,...,...,...,...,...,...
6553,2020-12-25,13.500707,916.151180,2.152634,261.457477,95.470006,32.118758
6554,2020-12-26,12.901577,917.194971,2.312996,253.686322,95.361429,32.068009
6555,2020-12-27,12.950693,916.302854,2.260313,252.095788,94.854902,34.539327
6556,2020-12-28,10.830827,916.237697,2.003774,254.229795,93.468912,31.310399


In [30]:
pune_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,7.100958,941.961323,0.608896,327.505071,91.114710,34.246204
1,2003-01-02,10.484796,942.675433,3.006925,290.310937,91.934631,35.734294
2,2003-01-03,12.606104,942.881457,2.872020,279.921010,92.393007,32.590960
3,2003-01-04,12.646417,943.009227,2.678422,274.952584,92.312594,35.435619
4,2003-01-05,13.066706,943.762527,3.104962,275.257170,92.569458,34.235347
...,...,...,...,...,...,...,...
6553,2020-12-25,12.096190,941.053885,2.152634,261.457477,92.849653,33.209944
6554,2020-12-26,13.381471,942.263093,2.312996,253.686322,93.400294,31.352555
6555,2020-12-27,13.819016,941.287692,2.260313,252.095788,93.560516,33.862652
6556,2020-12-28,13.831078,940.630939,2.003774,254.229795,93.895661,32.318171


In [31]:
hyd_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,9.801921,952.321702,2.702138,228.989644,91.137446,35.253027
1,2003-01-02,12.894156,953.896739,2.974241,287.669929,92.938629,31.660535
2,2003-01-03,16.945643,954.471306,3.015529,308.536731,96.690377,30.085849
3,2003-01-04,17.761199,954.172469,1.865083,299.395258,98.016959,30.569864
4,2003-01-05,14.281283,954.395644,2.287668,265.010524,93.245268,31.878895
...,...,...,...,...,...,...,...
6553,2020-12-25,13.330685,951.670802,2.015309,292.172202,94.673562,30.396700
6554,2020-12-26,14.027632,952.521458,1.736367,284.297801,94.788311,32.262344
6555,2020-12-27,13.223768,951.342613,1.754041,288.307250,94.039085,33.796627
6556,2020-12-28,12.504071,951.101661,2.012608,299.893708,93.248170,31.748070


# Since we will be applying a RNN, we will need the Date column to be Datetime column

In [32]:
blr_df['Date'] = pd.to_datetime(blr_df['Date'])
blr_df = blr_df.sort_values('Date')

In [33]:
hyd_df['Date'] = pd.to_datetime(hyd_df['Date'])
hyd_df = hyd_df.sort_values('Date')

In [34]:
pune_df['Date'] = pd.to_datetime(pune_df['Date'])
pune_df = pune_df.sort_values('Date')

# Splitting the Input and Predictor Variables

In [35]:
features = ['AP', 'DPT', 'WS', 'WSD', 'RH']
target = 'LST'

In [36]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [37]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [38]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the data

In [39]:
scaler = StandardScaler()

In [40]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Creating the sequences for RNN

In [41]:
def create_sequences(X, y, seq_length=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])  # predict LST at t+1
    return np.array(X_seq), np.array(y_seq)

In [42]:
seq_len = 30

In [43]:
blr_X_seq, blr_y_seq = create_sequences(blr_X_scaled, blr_y, seq_len)
hyd_X_seq, hyd_y_seq = create_sequences(hyd_X_scaled, hyd_y, seq_len)
pune_X_seq, pune_y_seq = create_sequences(pune_X_scaled, pune_y, seq_len)

# Splitting in Training and Testing Data

In [44]:
split_idx = int(0.8 * len(blr_X_seq))
blr_X_train, blr_X_test = blr_X_seq[:split_idx], blr_X_seq[split_idx:]
blr_y_train, blr_y_test = blr_y_seq[:split_idx], blr_y_seq[split_idx:]

In [45]:
split_idx = int(0.8 * len(hyd_X_seq))
hyd_X_train, hyd_X_test = hyd_X_seq[:split_idx], hyd_X_seq[split_idx:]
hyd_y_train, hyd_y_test = hyd_y_seq[:split_idx], hyd_y_seq[split_idx:]

In [46]:
split_idx = int(0.8 * len(pune_X_seq))
pune_X_train, pune_X_test = pune_X_seq[:split_idx], pune_X_seq[split_idx:]
pune_y_train, pune_y_test = pune_y_seq[:split_idx], pune_y_seq[split_idx:]

# Converting it into Pytorch Tensor for Further Analysis

In [47]:
blr_X_train_tensor  = torch.tensor(blr_X_train, dtype=torch.float32).to(device)
blr_y_train_tensor  = torch.tensor(blr_y_train, dtype=torch.float32).unsqueeze(1).to(device)
blr_X_test_tensor   = torch.tensor(blr_X_test, dtype=torch.float32).to(device)
blr_y_test_tensor   = torch.tensor(blr_y_test, dtype=torch.float32).unsqueeze(1).to(device)

hyd_X_train_tensor  = torch.tensor(hyd_X_train, dtype=torch.float32).to(device)
hyd_y_train_tensor  = torch.tensor(hyd_y_train, dtype=torch.float32).unsqueeze(1).to(device)
hyd_X_test_tensor   = torch.tensor(hyd_X_test, dtype=torch.float32).to(device)
hyd_y_test_tensor   = torch.tensor(hyd_y_test, dtype=torch.float32).unsqueeze(1).to(device)

pune_X_train_tensor = torch.tensor(pune_X_train, dtype=torch.float32).to(device)
pune_y_train_tensor = torch.tensor(pune_y_train, dtype=torch.float32).unsqueeze(1).to(device)
pune_X_test_tensor  = torch.tensor(pune_X_test, dtype=torch.float32).to(device)
pune_y_test_tensor  = torch.tensor(pune_y_test, dtype=torch.float32).unsqueeze(1).to(device)

# Defining the ANN Model

In [48]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)Wha

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # last timestep output
        return out

# Training the Model with the Hyperparameter Tuning done using Optuna

In [49]:
def objective(trial, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor):
    print("Model initialized, training starting...")
    hidden_size = trial.suggest_int("hidden_size", 50, 500)
    num_layers = trial.suggest_int("num_layers", 1, 2)
    dropout = trial.suggest_float("dropout", 0.0, 0.4) if num_layers > 1 else 0.0
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2, 4, 8, 16, 32, 64])
    epochs = trial.suggest_int("epochs", 50, 150)

    model = LSTMModel(X_train_tensor.shape[2], hidden_size, num_layers, dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        if epoch % 10 == 0:
            print(f"[Trial {trial.number}] Epoch {epoch}/{epochs}")
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        preds = model(X_test_tensor)
        mse = criterion(preds, y_test_tensor).item()
    return mse

# Applying the Model for Bengaluru

In [ ]:
blr_study = optuna.create_study(direction="minimize")
blr_study.optimize(lambda trial: objective(trial, blr_X_train_tensor, blr_y_train_tensor, blr_X_test_tensor, blr_y_test_tensor), n_trials=30)

[I 2025-04-21 18:00:19,862] A new study created in memory with name: no-name-b8926051-fdd7-446d-a784-190f6c3e23c8


Model initialized, training starting...
[Trial 0] Epoch 0/90
[Trial 0] Epoch 10/90
[Trial 0] Epoch 20/90
[Trial 0] Epoch 30/90
[Trial 0] Epoch 40/90
[Trial 0] Epoch 50/90
[Trial 0] Epoch 60/90
[Trial 0] Epoch 70/90
[Trial 0] Epoch 80/90


[I 2025-04-21 18:01:20,490] Trial 0 finished with value: 25.065479278564453 and parameters: {'hidden_size': 248, 'num_layers': 2, 'dropout': 0.09837676322021834, 'lr': 0.0028289739538963814, 'batch_size': 64, 'epochs': 90}. Best is trial 0 with value: 25.065479278564453.


Model initialized, training starting...
[Trial 1] Epoch 0/116
[Trial 1] Epoch 10/116
[Trial 1] Epoch 20/116
[Trial 1] Epoch 30/116
[Trial 1] Epoch 40/116
[Trial 1] Epoch 50/116
[Trial 1] Epoch 60/116
[Trial 1] Epoch 70/116
[Trial 1] Epoch 80/116


In [ ]:
blr_best_trial = blr_study.best_trial

In [ ]:
print(f"Best MSE for Bengaluru : {blr_best_trial.value:.4f}")
print(f"Best Parameters for Bengaluru : {blr_best_trial.params}\n")

# Applying for Hyderabad

In [ ]:
hyd_study = optuna.create_study(direction="minimize")
hyd_study.optimize(lambda trial: objective(trial, hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_test_tensor, hyd_y_test_tensor), n_trials=30)

In [ ]:
hyd_best_trial = hyd_study.best_trial

In [ ]:
print(f"4Best MSE for Hyderabad : {hyd_best_trial.value:.4f}")
print(f"Best Parameters for Hyderabad : {hyd_best_trial.params}\n")

# Applying for Pune

In [ ]:
pune_study = optuna.create_study(direction="minimize")
pune_study.optimize(lambda trial: objective(trial, pune_X_train_tensor, pune_y_train_tensor, pune_X_test_tensor, pune_y_test_tensor), n_trials=30)

In [ ]:
pune_best_trial = pune_study.best_trial

In [ ]:
print(f"Best MSE for Pune : {pune_best_trial.value:.4f}")
print(f"Best Parameters for Pune : {pune_best_trial.params}\n")

# Evaluating the Model and Visualizing the Results

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
def evaluate_predictions(y_true, y_pred):
    y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    
    # RSR = RMSE / STDEV of observed
    rsr = rmse / np.std(y_true)
    
    # PBIAS
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "MSE": mse,
        "MAE": mae,
        "R²": r2,
        "NSE": nse,
        "RSR": rsr,
        "PBIAS": pbias
    }

In [ ]:
def plot_predictions(y_true, y_pred, title="Prediction vs Ground Truth"):
    plt.figure(figsize=(10, 5))
    plt.plot(y_true.squeeze(), label="True", alpha=0.7)
    plt.plot(y_pred.squeeze(), label="Predicted", alpha=0.7)
    plt.title(title)
    plt.xlabel("Time Step")
    plt.ylabel("LST")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history

def plot_optuna_loss(study, title="Optuna Loss Over Trials"):
    fig = plot_optimization_history(study)
    fig.gca().set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
def train_final_model(X_train, y_train, X_test, y_test, best_params):
    model = LSTMModel(
        input_size=X_train.shape[2],
        hidden_size=best_params["hidden_size"],
        num_layers=best_params["num_layers"],
        dropout=best_params.get("dropout", 0.0)
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    criterion = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_train, y_train)
    loader = torch.utils.data.DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model.train()
    for epoch in range(best_params["epochs"]):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    return model

In [ ]:
import plotly.graph_objects as go
def plot_predictions_plotly(y_true, y_pred, title="Prediction vs Ground Truth (Test Data)"):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=y_true.squeeze(),
        mode='lines',
        name='True',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        y=y_pred.squeeze(),
        mode='lines',
        name='Predicted',
        line=dict(color='orange')
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Time Step (Test Set)",
        yaxis_title="LST",
        legend=dict(x=0.01, y=0.99),
        template='plotly_white',
        height=500,
        width=1000
    )

    fig.show()

In [ ]:
blr_best_params   = blr_best_trial.params
pune_best_params  = pune_best_trial.params
hyd_best_params   = hyd_best_trial.params

In [ ]:
blr_model = train_final_model(
    blr_X_train_tensor, blr_y_train_tensor,
    blr_X_test_tensor, blr_y_test_tensor,
    blr_best_params
)

In [ ]:
blr_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_blr = model(blr_X_test_tensor).cpu().numpy()
    y_true_blr = blr_y_test_tensor.cpu().numpy()

In [ ]:
metrics_blr = evaluate_predictions(y_true_blr, y_pred_blr)
print("Bangalore Metrics:")
for k, v in metrics_blr.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_blr, y_pred_blr, title="Bangalore's Prediction vs Ground Truth (Test)")

In [ ]:
hyd_model = train_final_model(
    hyd_X_train_tensor, hyd_y_train_tensor,
    hyd_X_test_tensor, hyd_y_test_tensor,
    hyd_best_params
)

In [ ]:
hyd_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_hyd = model(hyd_X_test_tensor).cpu().numpy()
    y_true_hyd = hyd_y_test_tensor.cpu().numpy()

In [ ]:
metrics_hyd = evaluate_predictions(y_true_hyd, y_pred_hyd)
print("Hyderabad Metrics:")
for k, v in metrics_hyd.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_hyd, y_pred_hyd, title="Hyderabad's Prediction vs Ground Truth (Test)")

In [ ]:
pune_model = train_final_model(
    pune_X_train_tensor, pune_y_train_tensor,
    pune_X_test_tensor, pune_y_test_tensor,
    pune_best_params
)

In [ ]:
pune_model.eval()

In [ ]:
with torch.no_grad():
    y_pred_pune = model(pune_X_test_tensor).cpu().numpy()
    y_true_pune = pune_y_test_tensor.cpu().numpy()

In [ ]:
metrics_pune = evaluate_predictions(y_true_pune, y_pred_pune)
print("Pune's Metrics:")
for k, v in metrics_pune.items():
    print(f"{k}: {v:.4f}")

In [ ]:
plot_predictions_plotly(y_true_pune, y_pred_pune, title="Pune's Prediction vs Ground Truth (Test)")

# Storing the model for future use

In [ ]:
torch.save(blr_model.state_dict(), "blr_rnn_model.pth")

In [ ]:
torch.save(hyd_model.state_dict(), "hyd_rnn_model.pth")

In [ ]:
torch.save(pune_model.state_dict(), "pune_rnn_model.pth")